In [5]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import Annotated
from typing_extensions import TypedDict
from operator import add
from langchain_google_genai import ChatGoogleGenerativeAI

from dotenv import load_dotenv

In [6]:
load_dotenv()

True

In [7]:
def extract_text(response) -> str:
    """ Normalizes an LLM response's content into a plain string.

    Some models (e.g. Gemini) can return `content` as a list of content
    blocks (dicts with a 'text' key, or plain strings) instead of a str.
    """
    if not response:
        return "No summary available."

    content = response.content
    if isinstance(content, list):
        text = "".join(
            part.get("text", "") if isinstance(part, dict) else str(part)
            for part in content
        )
    else:
        text = content

    return text.strip() if text else "No summary available."

In [8]:

class SharedState(TypedDict):
    dsa_topics: list[str]
    system_design_topics: list[str]

def parse_model_response(response):
    """ Parse the model response"""
    return [topic.strip() for topic in response.split(",") if topic.strip()]

In [ ]:
def find_relevant_dsa_topics(shared_state: SharedState) -> SharedState:
    """ Find relevant DSA topics for year 2025."""
    query = """
    Can you provide top 5 DSA topics to master for Software Engineering interviews in 2025?.
    Please only return the DSA topics comma separated, no other detail is needed. 
    """
    model = ChatGoogleGenerativeAI(model="gemini-3-flash-preview", temperature=0)
    
    response = model.invoke(query)
    # shared_state["dsa_topics"] = parse_model_response(response.content)

    return {
        "dsa_topics": parse_model_response(extract_text(response)),
        "system_design_topics": []
    }

In [10]:
def find_relevant_system_design_topics(shared_state: SharedState) -> SharedState:
    """ Find relevant System Design topics for year 2025."""
    query = """
    Can you provide top 5 System Design topics to master for Software Engineering interviews in 2025?.
    Please only return the System Design topics comma separated, no other detail is needed. 
    """
    model = ChatGoogleGenerativeAI(model="gemini-3-flash-preview", temperature=0)
    
    response = model.invoke(query)
    # shared_state["system_design_topics"] = parse_model_response(response.content)

    return {
        "dsa_topics": [],
        "system_design_topics": parse_model_response(extract_text(response))
    }

In [11]:
def build_graph():
    # Building a Graph
  # State of the Graph that will be shared among nodes.
  workflow = StateGraph(SharedState)

  # Add nodes.
  workflow.add_node("find_relevant_dsa_topics", find_relevant_dsa_topics)
  workflow.add_node("find_relevant_system_design_topics", find_relevant_system_design_topics)

  # Define the edges of the graph.
  workflow.add_edge(START, "find_relevant_dsa_topics")
  workflow.add_edge("find_relevant_dsa_topics", "find_relevant_system_design_topics")
  workflow.add_edge("find_relevant_system_design_topics", END)

  checkpointer = InMemorySaver()
  graph = workflow.compile(checkpointer=checkpointer)

  config = {"configurable": {"thread_id": "1"}}
  response = graph.invoke({}, config)

  # print(graph.get_graph().draw_mermaid())

  return response, graph, config

In [12]:
response, graph, config = build_graph()
print(response)

{'dsa_topics': [], 'system_design_topics': ['Scalability and Load Balancing', 'Database Sharding and Replication', 'Caching Strategies', 'Microservices and API Design', 'Message Queues and Event-Driven Architecture']}


In [13]:
snapshots = list(graph.get_state_history(config))

In [14]:
list.reverse(snapshots)

for snapshot in snapshots:
    print("================== Snapshot details: ==================")
    print(snapshot.values)
    print(snapshot.next)
    print("\n\n")

================== Snapshot details: ==================
{}
('__start__',)



================== Snapshot details: ==================
{}
('find_relevant_dsa_topics',)



================== Snapshot details: ==================
{'dsa_topics': ['Arrays and Hashing', 'Two Pointers and Sliding Window', 'Trees and Graphs', 'Dynamic Programming', 'Heaps and Priority Queues'], 'system_design_topics': []}
('find_relevant_system_design_topics',)



================== Snapshot details: ==================
{'dsa_topics': [], 'system_design_topics': ['Scalability and Load Balancing', 'Database Sharding and Replication', 'Caching Strategies', 'Microservices and API Design', 'Message Queues and Event-Driven Architecture']}
()



